In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
import numpy as np
import pandas as pd
from twiga.core.plot.matplot_theme import configure_matlib_style

configure_matlib_style(latex=True)
matplotlib_inline.backend_inline.set_matplotlib_formats("svg", "pdf", "retina")  # For export

In [ ]:
from twiga.core.metrics.cwe import ArrayLike, cwe_score, gamma_coverage_score, gamma_nmpi_score

In [ ]:
ig, ax1 = plt.subplots(1, 1, figsize=(3, 2.5))
pics = np.linspace(0.0, 1, 20)
ax1.set_ylim(0, 1.0)
ax1.set_xlim(0, 1.0)
ax1.plot(pics, gamma_coverage_score(picp=pics, alpha=0.05), label=r"$\Gamma^E=0.05$")
ax1.plot(pics, gamma_coverage_score(picp=pics, alpha=0.1), label=r"$\Gamma^E=0.1$")
ax1.plot(pics, gamma_coverage_score(picp=pics, alpha=0.2), label=r"$\Gamma^E=0.2$")
ax1.set_ylabel(r"$\gamma_{coverage}$")
ax1.set_xlabel(r"$\mathrm{PCIP}$")
ax1.legend()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(3, 2.5))
true_nmpic = 0.3
nrmse = 0.09
pics = np.linspace(0, 1, 10)
nmpis = np.linspace(0.0, 1, 20)
ax1.set_ylim(0, 1)
ax1.set_xlim(0, 1)

ax1.plot(nmpis, gamma_nmpi_score(nmpi_val=nmpis, true_nmpic=0.25), label=r"$\Gamma^E=0.01$")
ax1.plot(nmpis, gamma_nmpi_score(nmpi_val=nmpis, true_nmpic=0.10), label=r"$\Gamma^E=0.09$")
ax1.plot(nmpis, gamma_nmpi_score(nmpi_val=nmpis, true_nmpic=0.3), label=r"$\Gamma^E=0.2$")
ax1.set_ylabel(r"$\gamma_{nmpi}$")
ax1.set_xlabel(r"$\mathrm{NMPI}$")
ax1.legend()

In [ ]:
pics = np.array([0.5, 0.80, 0.95])
nmpi = 0.3
nmrmses = np.linspace(0.01, 1, 20)
scores = [[cwe_score(nmpi, pic, nrmse, true_nmpic=true_nmpic) for nrmse in nmrmses] for pic in pics]
fig, ax1 = plt.subplots(1, 1, figsize=(3, 2.5))
for i, pic in enumerate(pics):
    ax1.plot(nmrmses, scores[i], label=f"PIC:={round(pic, 3)}")
ax1.set_ylim(0, 1)
ax1.set_xlim(0, 1)
ax1.set_xlabel("NMRSE")
ax1.set_ylabel("CWE")

In [ ]:
nmpis = np.linspace(0.01, 1, 20)
scores = [[cwe_score(nmpi, pic, nrmse, true_nmpic=true_nmpic) for nmpi in nmpis] for pic in pics]
fig, ax1 = plt.subplots(1, 1, figsize=(3, 2.5))
ax1.set_ylim(0, 1)
ax1.set_xlim(0, 1)
for i, pic in enumerate(pics):
    ax1.plot(nmpis, scores[i], label=f"PCIP:={round(pic, 3)}")
ax1.set_xlabel("NMPI")
ax1.set_ylabel("CWE")
ax1.legend()

In [ ]:
nrmse = 0.09
pics = np.linspace(0, 1, 20)
nmpis = np.array([0.2, 0.35, 0.8])
scores = [[cwe_score(nmpi, pic, nrmse, true_nmpic=true_nmpic) for pic in pics] for nmpi in nmpis]
fig, ax1 = plt.subplots(1, 1, figsize=(3, 2.5))
for i, nmpi in enumerate(nmpis):
    ax1.plot(pics, scores[i], linestyle="solid", label=f"NMP:={round(nmpi, 3)}")
ax1.set_xlabel("PICP")
ax1.set_ylabel("CWE")
ax1.legend()
ax1.set_ylim(0, 1)
ax1.set_xlim(0, 1)

In [ ]:
from twiga.core.metrics.interval import _kappa_spread, _nmpi_scale_factor, get_interval_metrics, interval_coverage, nmpi

In [ ]:
def plot_interval_metrics(
    interval_scale: int = 2, alpha: float = 0.1, nmpi_scale: str = "max", spread: str = "std", axis: int = 0
) -> plt.Figure:
    """Plot interval metrics for sinusoidal ground truth data with predictions and good intervals.

    This function generates synthetic sinusoidal true values, adds noise for predictions,
    computes prediction intervals, calculates metrics like PICP, NMPI, and CWE score,
    and returns a matplotlib figure with the plot.

    Args:
    interval_scale : int, default=2
        Scale factor for capping the interval half-width to keep NMPI ≤ 1.
        The half-width is min(std(y), y_range / interval_scale).
    alpha : float, default=0.1
        Significance level for coverage in CWE score calculation.
    nmpi_scale : str, default="max"
        Scaling method for NMPI normalization (e.g., "max" for max value).
    spread : str, default="std"
        Measure of spread for true NMPI calculation (e.g., "std" for standard deviation).
    axis : int, default=0
        Axis along which to compute metrics.

    Returns:
    fig : matplotlib.figure.Figure
        The matplotlib figure containing the plot of true values, predictions, and intervals.
    """
    np.random.seed(42)
    t = np.linspace(0, 4 * np.pi, 200)
    y = 50 + 10 * np.sin(t)
    yhat = y + np.random.normal(0, 0.1, 200)
    y_range = y.max() - y.min()

    optimal_half_width = np.std(y)
    # Cap the half-width to keep NMPI ≤ 1
    half_good = min(optimal_half_width, y_range / interval_scale)
    lower_good = yhat - half_good
    upper_good = yhat + half_good
    true_arr, lower_arr, upper_arr = np.asarray(y), np.asarray(lower_good), np.asarray(upper_good)
    picp_val = interval_coverage(true_arr, lower_arr, upper_arr, axis=axis)
    scale_ref = _nmpi_scale_factor(true_arr, nmpi_scale, axis=None)
    nmpi_val = nmpi(lower_arr, upper_arr, scale=scale_ref, axis=axis)

    kappa = _kappa_spread(true_arr, spread, axis=None)
    true_nmpic = np.divide(kappa, scale_ref, out=np.zeros_like(np.asarray(kappa, dtype=float)), where=scale_ref != 0)
    score = cwe_score(nmpi_val=nmpi_val, picp=picp_val, error=None, true_nmpic=true_nmpic, alpha=alpha)

    fig, ax1 = plt.subplots(1, 1, figsize=(4, 2.5))
    ax1.plot(t, y, label="True Values", alpha=0.5)
    ax1.plot(t, yhat, label="Predictions", alpha=0.5)
    ax1.plot(t, lower_good, color="green", label="Good Interval", alpha=0.7)
    ax1.plot(t, upper_good, color="green", alpha=0.7)
    ax1.set_title(
        f"CWE Score: {score[0]:.1f}, NMPI: {nmpi_val:.3f},  \n  PICP: {picp_val:.1f}  True NMPI: {true_nmpic:.3f}"
    )
    ax1.set_xlabel("Time")
    ax1.set_ylabel("Values")
    ax1.legend()
    print(f"CWE Score: {score[0]:.1f}, NMPI: {nmpi_val:.3f}, PICP: {picp_val:.1f}, True NMPI: {true_nmpic:.3f}")
    return fig

In [ ]:
fig = plot_interval_metrics(interval_scale=20)

In [ ]:
fig = plot_interval_metrics(interval_scale=10)

In [ ]:
fig = plot_interval_metrics(interval_scale=5)

## Soundness: is the metric optimised by the honest forecast?

A metric is only useful for ranking if the *true* forecast wins. Sweep a Gaussian
family through the truth and see where each metric peaks.

In [ ]:
import sys

from chapters.ch3_methodology.cwe_validation.metric_definitions import soundness_sweep

for mode in ("std", "iqr", "mad", "nominal"):
    r = soundness_sweep(alpha=0.10, kappa_mode=mode)
    tag = "SOUND" if abs(r["cwe_argopt"] - 1.0) <= 0.02 else "unsound"
    print(
        f"kappa={mode:<8} ref={r['ref']:.4f}  CRPS={r['crps_argopt']:.3f} "
        f"WS={r['winkler_argopt']:.3f}  CWE={r['cwe_argopt']:.3f}  {tag}"
    )

`kappa` must be tied to `alpha`. A calibrated 90 % interval is 3.29σ wide, but
`std` targets 1.00σ — so the width term pulls toward intervals ~3× too narrow and the
optimum lands on an over-confident forecast. The **nominal width**
`q_{1-a/2} - q_{a/2}` makes both CWE components peak at the same place.

## Conditional blindness, and the fix

Marginal CWE aggregates widths *before* comparing to the reference, so a constant-width
forecaster can hide compensating errors. Here `y | x ~ N(0, s(x))`:

* **A** — width tracks the true `s(x)` (correct)
* **B** — one constant width, marginally calibrated (wrong)

In [ ]:
import numpy as np
from scipy import stats as st
from twiga.core.metrics.cwe import cwe_score
from twiga.core.metrics.interval import conditional_cwe, nmpi, winkler_score

ALPHA = 0.10
z = st.norm.ppf(1 - ALPHA / 2)
rng = np.random.default_rng(0)
n = 200_000
sx = rng.uniform(0.3, 3.0, n)
y = rng.normal(0, sx)
R = float(y.max() - y.min())

lo_a, up_a = -z * sx, z * sx  # A: conditionally correct
k = np.quantile(y, 1 - ALPHA / 2)
lo_b, up_b = np.full(n, -k), np.full(n, k)  # B: constant width

edges = np.quantile(sx, np.linspace(0, 1, 6))
edges[-1] += 1e-9
bins = np.clip(np.digitize(sx, edges[1:-1]), 0, 4)
print(f"{'s(x) quintile':<16}{'PICP A':>9}{'PICP B':>9}   (nominal 0.900)")
for i in range(5):
    m = bins == i
    print(
        f"{i + 1} (s~{sx[m].mean():.2f}){'':<6}"
        f"{interval_coverage(y[m], lo_a[m], up_a[m]):>9.3f}"
        f"{interval_coverage(y[m], lo_b[m], up_b[m]):>9.3f}"
    )
print(
    f"{'marginal':<16}{interval_coverage(y, lo_a, up_a):>9.3f}{interval_coverage(y, lo_b, up_b):>9.3f}   <- identical"
)

In [ ]:
def cwe_marginal(y_, lo, up, err=None):
    ref = float(np.quantile(y_, 1 - ALPHA / 2) - np.quantile(y_, ALPHA / 2)) / R
    return float(
        np.ravel(cwe_score(float(nmpi(lo, up, scale=R)), float(interval_coverage(y_, lo, up)), err, ref, alpha=ALPHA))[
            0
        ]
    )


err = float(np.mean(np.abs(y))) / R
rows = [
    (
        "Winkler (lower better)",
        winkler_score(y, lo_a, up_a, alpha=ALPHA),
        winkler_score(y, lo_b, up_b, alpha=ALPHA),
        "lower",
    ),
    ("CWE marginal", cwe_marginal(y, lo_a, up_a, err), cwe_marginal(y, lo_b, up_b, err), "higher"),
    (
        "CWE conditional (FIX)",
        conditional_cwe(y, lo_a, up_a, sx, alpha=ALPHA, error=err),
        conditional_cwe(y, lo_b, up_b, sx, alpha=ALPHA, error=err),
        "higher",
    ),
]
print(f"{'metric':<26}{'A correct':>11}{'B wrong':>10}   picks")
for name, a, b, orient in rows:
    good = a < b if orient == "lower" else a > b
    print(f"{name:<26}{a:>11.4f}{b:>10.4f}   {'A  correct' if good else 'B  WRONG'}")

The marginal form is not merely wrong here — it is **unstable**. Its verdict is decided
by whether the sharpness-bonus gate fires at `PICP >= 1-alpha`, which turns on the fourth
decimal place of PICP. Across seeds it picks the wrong forecaster about half the time.

In [ ]:
# Across seeds the marginal form flips; the conditional form does not.
def marginal_cwe(y_, lo, up, scale_r, err):
    """Marginal CWE with an alpha-tied reference (no closure over loop state)."""
    ref = float(np.quantile(y_, 1 - ALPHA / 2) - np.quantile(y_, ALPHA / 2)) / scale_r
    return float(
        np.ravel(
            cwe_score(
                float(nmpi(lo, up, scale=scale_r)),
                float(interval_coverage(y_, lo, up)),
                err,
                ref,
                alpha=ALPHA,
            )
        )[0]
    )


for seed in range(6):
    g = np.random.default_rng(seed)
    sx_s = g.uniform(0.3, 3.0, n)
    y_s = g.normal(0, sx_s)
    r_s = float(y_s.max() - y_s.min())
    la, ua = -z * sx_s, z * sx_s
    k_s = np.quantile(y_s, 1 - ALPHA / 2)
    lb, ub = np.full(n, -k_s), np.full(n, k_s)
    e_s = float(np.mean(np.abs(y_s))) / r_s

    ma = marginal_cwe(y_s, la, ua, r_s, e_s)
    mb = marginal_cwe(y_s, lb, ub, r_s, e_s)
    ca = conditional_cwe(y_s, la, ua, sx_s, alpha=ALPHA, error=e_s)
    cb = conditional_cwe(y_s, lb, ub, sx_s, alpha=ALPHA, error=e_s)
    print(
        f"seed {seed}  marginal {ma:.4f}/{mb:.4f} -> {'A' if ma > mb else 'B  WRONG':<8}"
        f"   conditional {ca:.4f}/{cb:.4f} -> {'A' if ca > cb else 'B  WRONG'}"
    )

**Why the fix works.** `conditional_cwe` scores *within* each stratum and averages
afterwards, so compensating errors no longer cancel. The normaliser `R` is computed once
over the whole sample and shared across strata — recomputing it per stratum would rescale
NMPI differently in each bin and make them incomparable.

**Caveats.** Stratification needs a conditioning variable available at forecast time (lead
time is the natural choice here, and Chapter 7's per-horizon calibration already exploits
it). And this does not make CWE a proper scoring rule — it is simply no longer blind to
the failure mode that dominates LV net-load. CRPS and the interval score still carry the
conclusions.